# Демонстрация отнесения результатов анализов к группам риска

Демонстрируется работа модуля `model/stat.py` по отнесению результатов анализа к трем группам риска:
- "высокий риск, требуется диагностика у врача" (high risk),
- "имеются некоторые признаки, рекомендуется позже повторить проверку" (worry about),
- "не выявлено нарушений в значимом объеме" (no risk).

## Набор тестов

**На наборах пар аудио-текст**: обрабатываем три группы пар аудио-текст, отнесенными к трем группам риска

## Пороги в долях ошибок

| Группа риска | Количество ошибок | Порог |
|------------|------------|----------|
| no_risk | 0-3.3% | errors <= total/30 |
| worry | 3.3-7% | total/30 < errors < total*0.07 |
| high_risk | >=7% | errors >= total*0.07 |

## Структура тестовых данных

```
test_data/
└── classification/
    ├── high_risk/
    │   ├── audio1.wav, text1.txt
    │   └── ...
    ├── worry/
    │   ├── audio1.wav, text1.txt
    │   └── ...
    └── no_risk/
        ├── audio1.wav, text1.txt
        └── ...
```

## 1. Импорты

In [1]:
import importlib
import sys
import os
import json
from pathlib import Path
from IPython.display import JSON, display

# Add model directory to path
sys.path.insert(0, '../model')

# Import libraries
# Remove from cache if it exists
if 'statdiag' in sys.modules:
    del sys.modules['statdiag']
import statdiag as stat_module
importlib.reload(stat_module)

# Remove from cache if it exists
if 'dysmarkccls_clean' in sys.modules:
    del sys.modules['dysmarkccls_clean']
# Import dysmark module
import dysmarkccls_clean as dysmark
importlib.reload(dysmark)

print("Modules loaded successfully")
print(f"statdiag.py version: library mode available")
print(f"dysmark.py version: library mode available")

Modules loaded successfully
statdiag.py version: library mode available
dysmark.py version: library mode available


## 2. Классификация примеров аудиозаписей

Проверим классификацию трех подготовленных групп аудио.

In [2]:
# Configure test directories
BASE_DIR = Path('..')
CLASSIFICATION_DIR = BASE_DIR / 'test_data' / 'classification'

RISK_GROUPS = {
    'no_risk': CLASSIFICATION_DIR / 'no_risk',
    'worry': CLASSIFICATION_DIR / 'worry',
    'high_risk': CLASSIFICATION_DIR / 'high_risk'
}

print("Test directories:")
for group, path in RISK_GROUPS.items():
    exists = "EXISTS" if path.exists() else "NOT FOUND"
    print(f"  {group}: {path} [{exists}]")

Test directories:
  no_risk: ../test_data/classification/no_risk [EXISTS]
  worry: ../test_data/classification/worry [EXISTS]
  high_risk: ../test_data/classification/high_risk [EXISTS]


## 3. Подготовка списков пар аудио-текст

In [3]:
def discover_audio_text_pairs(directory: Path) -> list:
    """
    Find all matching audio-text pairs in a directory.
    
    Returns:
        List of tuples: (audio_path, text_path)
    """
    pairs = []
    
    if not directory.exists():
        print(f"Warning: Directory not found: {directory}")
        return pairs
    
    # Find all wav files
    audio_files = sorted(directory.glob('*.wav'))
    
    for audio_file in audio_files:
        # Find corresponding text file (same stem)
        stem = audio_file.stem
        text_file = directory / f'{stem}.txt'
        
        if text_file.exists():
            pairs.append((audio_file, text_file))
        else:
            print(f"Warning: No text file for {audio_file.name}")
    
    return pairs


# Discover pairs for each risk group
print("Discovering audio-text pairs...\n")

test_pairs = {}
for group, directory in RISK_GROUPS.items():
    pairs = discover_audio_text_pairs(directory)
    test_pairs[group] = pairs
    print(f"{group}: {len(pairs)} pairs found")

total_pairs = sum(len(p) for p in test_pairs.values())
print(f"\nTotal pairs: {total_pairs}")

Discovering audio-text pairs...

no_risk: 1 pairs found
worry: 1 pairs found
high_risk: 1 pairs found

Total pairs: 3


## 4. Поиск маркеров в аудиозаписях с помощью модуля dysmarkccls_clean.py

In [4]:
def process_risk_group(group_name: str, pairs: list) -> dict:
    """
    Process all audio-text pairs for a risk group through dysmark.
    
    Returns:
        Dictionary with analyses and metadata
    """
    print(f"\n{'=' * 80}")
    print(f"PROCESSING: {group_name}")
    print(f"{'=' * 80}")
    
    analyses = []
    quality_stats = {'good': 0, 'bad': 0}
    
    for i, (audio_path, text_path) in enumerate(pairs, 1):
        print(f"\n[{i}/{len(pairs)}] {audio_path.name}")
        
        try:
            # Read reference text
            with open(text_path, 'r', encoding='utf-8') as f:
                reference_text = f.read().strip()
            
            # Process through dysmark with quality check
            result = dysmark.get_markers(reference_text, str(audio_path))
            
            # Check quality
            if result.get('quality') == 'good':
                analyses.append({"triples": result['triples']})
                quality_stats['good'] += 1
                print(f"  Status: GOOD quality")
            else:
                quality_stats['bad'] += 1
                print(f"  Status: BAD quality - {result.get('problem', 'unknown')}")
                
        except Exception as e:
            quality_stats['bad'] += 1
            print(f"  Error: {str(e)}")
    
    return {
        'group_name': group_name,
        'analyses': analyses,
        'quality_stats': quality_stats,
        'total_files': len(pairs)
    }


# Process all risk groups
print("=" * 80)
print("PROCESSING AUDIO FILES THROUGH DYSMARK")
print("=" * 80)
print("\nNote: Model will load on first use (may take 30-60 seconds)")

group_results = {}
for group_name, pairs in test_pairs.items():
    if pairs:
        group_results[group_name] = process_risk_group(group_name, pairs)

print("\n" + "=" * 80)
print("AUDIO PROCESSING COMPLETE")
print("=" * 80)

PROCESSING AUDIO FILES THROUGH DYSMARK

Note: Model will load on first use (may take 30-60 seconds)

PROCESSING: no_risk

[1/1] fruits.wav


Loading model (first use)...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Model loaded.


  Status: GOOD quality

PROCESSING: worry

[1/1] worry.wav
  Status: GOOD quality

PROCESSING: high_risk

[1/1] high_risk.wav
  Status: GOOD quality

AUDIO PROCESSING COMPLETE


In [5]:
group_results

{'no_risk': {'group_name': 'no_risk',
  'analyses': [{'triples': [['ф', 'ф', '0'],
     ['р', 'р', '0'],
     ['у', 'у', '0'],
     ['к', 'к', '0'],
     ['т', 'т', '0'],
     ['ы', 'ы', '0']]}],
  'quality_stats': {'good': 1, 'bad': 0},
  'total_files': 1},
 'worry': {'group_name': 'worry',
  'analyses': [{'triples': [['|', 'м', 'J'],
     ['н', 'н', '0'],
     ['е', 'е', '0'],
     ['|', '|', '0'],
     ['т', 'т', '0'],
     ['о', 'о', '0'],
     ['ж', 'ж', '0'],
     ['е', 'е', '0'],
     ['|', '|', '0'],
     ['з', 'з', '0'],
     ['а', 'а', '0'],
     ['х', 'х', '0'],
     ['о', 'о', '0'],
     ['т', 'т', '0'],
     ['е', 'е', '0'],
     ['л', 'л', '0'],
     ['о', 'о', '0'],
     ['с', 'с', '0'],
     ['ь', 'ь', '0'],
     ['|', '|', '0'],
     ['|', 'с', 'J'],
     ['л', 'л', '0'],
     ['о', 'о', '0'],
     ['в', 'в', '0'],
     ['и', 'и', '0'],
     ['т', 'т', '0'],
     ['ь', 'ь', '0'],
     ['|', '|', '0'],
     ['р', 'р', '0'],
     ['а', 'а', '0'],
     ['к', 'к', '0'],
  

## 5. Классификация полученных анализов

In [6]:
def classify_group(group_result: dict) -> dict:
    """
    Run stat.py classification on a processed risk group.
    
    Returns:
        Classification results with expected vs actual comparison
    """
    group_name = group_result['group_name']
    analyses = group_result['analyses']
    
    # Create input for stat.diagnose()
    input_data = {"analyses": analyses}
    
    # Run classification
    classification = stat_module.diagnose(input_data)
    print("Classification")
    print(classification)
    
    # Determine expected risk group from directory name
    expected_map = {
        'no_risk': 'no risk',
        'worry': 'worry about',
        'high_risk': 'risk group'
    }
    expected = expected_map.get(group_name, 'unknown')
    
    # Check if classification matches expectation
    actual = classification['risk_group']
    passed = actual == expected
    
    return {
        'group_name': group_name,
        'expected': expected,
        'actual': actual,
        'passed': passed,
        'classification': classification,
        'quality_stats': group_result['quality_stats'],
        'total_files': group_result['total_files'],
        'files_analyzed': len(analyses)
    }


# Run classification for each group
print("\n" + "=" * 80)
print("RUNNING CLASSIFICATION")
print("=" * 80)

classification_results = []
for group_name, group_result in group_results.items():
    result = classify_group(group_result)
    classification_results.append(result)
    
    status = "PASS" if result['passed'] else "FAIL"
    print(f"\n{status} | {group_name}")
    print(f"  Expected: {result['expected']}")
    print(f"  Actual:   {result['actual']}")
    print(f"  Files analyzed: {result['files_analyzed']}/{result['total_files']}")
    print(f"  Quality: {result['quality_stats']['good']} good, {result['quality_stats']['bad']} bad")
    print(f"  Phonemes: {result['classification']['total_phonemes']}")
    print(f"  Markers: {result['classification']['total_markers']}")
    print(f"  Error rate: {result['classification']['total_markers']/result['classification']['total_phonemes']*100:.1f}%")
    print(f"  Norma threshold: {result['classification']['norma']}")


RUNNING CLASSIFICATION
Classification
{'total_phonemes': 6, 'total_markers': 0, 'marker_statistics': {}, 'norma': 1, 'risk_group': 'no risk'}

PASS | no_risk
  Expected: no risk
  Actual:   no risk
  Files analyzed: 1/1
  Quality: 1 good, 0 bad
  Phonemes: 6
  Markers: 0
  Error rate: 0.0%
  Norma threshold: 1
Classification
{'total_phonemes': 135, 'total_markers': 9, 'marker_statistics': {'J': 4, 'D': 4, 'C': 1}, 'norma': 5, 'risk_group': 'worry about'}

PASS | worry
  Expected: worry about
  Actual:   worry about
  Files analyzed: 1/1
  Quality: 1 good, 0 bad
  Phonemes: 135
  Markers: 9
  Error rate: 6.7%
  Norma threshold: 5
Classification
{'total_phonemes': 136, 'total_markers': 16, 'marker_statistics': {'J': 4, 'C': 6, '7': 1, 'H': 1, 'D': 2, 'K': 2}, 'norma': 5, 'risk_group': 'risk group'}

PASS | high_risk
  Expected: risk group
  Actual:   risk group
  Files analyzed: 1/1
  Quality: 1 good, 0 bad
  Phonemes: 136
  Markers: 16
  Error rate: 11.8%
  Norma threshold: 5


## 6. Сводная таблица результатов

In [7]:
# Display summary table
print("\n" + "=" * 80)
print("CLASSIFICATION TEST SUMMARY")
print("=" * 80)
print(f"{'Risk Group':<15} {'Expected':<15} {'Actual':<15} {'Status':<10} {'Erroneous Phonemes Rate':<12}")
print("-" * 80)

passed = sum(1 for r in classification_results if r['passed'])
total = len(classification_results)

for r in classification_results:
    status = "PASS" if r['passed'] else "FAIL"
    error_rate = r['classification']['total_markers']/r['classification']['total_phonemes']*100
    print(f"{r['group_name']:<15} {r['expected']:<15} {r['actual']:<15} {status:<10} {error_rate:.1f}%")

print("-" * 80)
print(f"Passed: {passed}/{total} ({passed/total*100:.0f}%)")
print("=" * 80)


CLASSIFICATION TEST SUMMARY
Risk Group      Expected        Actual          Status     Erroneous Phonemes Rate
--------------------------------------------------------------------------------
no_risk         no risk         no risk         PASS       0.0%
worry           worry about     worry about     PASS       6.7%
high_risk       risk group      risk group      PASS       11.8%
--------------------------------------------------------------------------------
Passed: 3/3 (100%)
